# Electric Truck TCO & Fleet Economics Analysis

**Portfolio project for Business & Strategy Analyst internship applications**

This notebook compares diesel and electric freight-truck economics using unit economics, TCO, payback, ROI proxy, fleet scenarios, and electricity-price sensitivity.

> **Important:** Published/reference inputs are separated from assumptions. Assumptions should be validated with OEM/dealer quotes and actual fleet operating data before making a real investment decision.

## 1. Project Objectives

- Compare diesel vs. electric truck operating economics.
- Calculate **₹/km** and **₹/ton-km**.
- Estimate annual OPEX and savings.
- Calculate simple payback period and a 5-year ROI proxy.
- Estimate economics for **10, 100 and 1,000 trucks**.
- Test how electricity-price changes affect EV economics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Input Data

The model intentionally distinguishes **reference/published data** from **scenario assumptions**.

In [ ]:
# -----------------------------
# Inputs
# -----------------------------
annual_km = 80_000
years = 5

# Reference / published inputs
diesel_price = 105.61                 # ₹/L; dated market reference
electricity_price = 8.27              # ₹/kWh; reference tariff input; validate actual charging tariff
ev_battery_kwh = 450                   # Tata Prima E.55S published option
ev_range_km = 350                       # Tata published 'up to' range

# Scenario assumptions
diesel_efficiency = 2.75              # km/L
charging_overhead = 0.15              # 15% charging-loss/overhead assumption
diesel_purchase = 4_049_000           # ₹; indicative third-party listing
ev_purchase = 6_500_000               # ₹; illustrative assumption
diesel_maintenance = 10               # ₹/km
ev_maintenance = 6                    # ₹/km
diesel_other_opex = 2                 # ₹/km
ev_other_opex = 2                     # ₹/km
payload_tonnes = 25                   # tonnes
charger_capex_per_ev = 500_000       # ₹/EV truck
residual_pct = 0.10                   # 10% residual value assumption

inputs = pd.DataFrame({
    'Parameter': [
        'Annual distance', 'Analysis period', 'Diesel price',
        'Diesel efficiency', 'Electricity price', 'EV battery capacity',
        'EV range', 'Charging overhead', 'Diesel purchase price',
        'EV purchase price', 'Diesel maintenance', 'EV maintenance',
        'Diesel other OPEX', 'EV other OPEX', 'Payload',
        'Charging CAPEX per EV', 'Residual value'
    ],
    'Value': [
        annual_km, years, diesel_price, diesel_efficiency, electricity_price,
        ev_battery_kwh, ev_range_km, charging_overhead, diesel_purchase,
        ev_purchase, diesel_maintenance, ev_maintenance,
        diesel_other_opex, ev_other_opex, payload_tonnes,
        charger_capex_per_ev, residual_pct
    ],
    'Type': [
        'Assumption', 'Assumption', 'Reference', 'Assumption', 'Reference',
        'Published', 'Published', 'Assumption', 'Reference/Listing',
        'Assumption', 'Assumption', 'Assumption', 'Assumption',
        'Assumption', 'Assumption', 'Assumption', 'Assumption'
    ]
})
inputs

## 3. Unit Economics Calculations

**TCO = Total Cost of Ownership**.

The model first calculates energy cost per kilometre, total operating cost per kilometre, and freight cost per ton-kilometre.

In [ ]:
# Energy cost per km
diesel_energy_cost_km = diesel_price / diesel_efficiency

# EV energy consumption proxy derived from published battery/range figures
ev_energy_km = (ev_battery_kwh / ev_range_km) * (1 + charging_overhead)
ev_energy_cost_km = electricity_price * ev_energy_km

# Total operating cost per km
diesel_opex_km = diesel_energy_cost_km + diesel_maintenance + diesel_other_opex
ev_opex_km = ev_energy_cost_km + ev_maintenance + ev_other_opex

# Freight unit economics
diesel_ton_km = diesel_opex_km / payload_tonnes
ev_ton_km = ev_opex_km / payload_tonnes

unit_economics = pd.DataFrame({
    'Metric': [
        'Energy cost / km', 'Total operating cost / km', 'Operating cost / ton-km'
    ],
    'Diesel': [diesel_energy_cost_km, diesel_opex_km, diesel_ton_km],
    'Electric': [ev_energy_cost_km, ev_opex_km, ev_ton_km]
})
unit_economics

In [ ]:
print(f'Diesel energy cost/km: ₹{diesel_energy_cost_km:,.2f}')
print(f'EV energy cost/km:     ₹{ev_energy_cost_km:,.2f}')
print(f'Diesel operating cost/km: ₹{diesel_opex_km:,.2f}')
print(f'EV operating cost/km:     ₹{ev_opex_km:,.2f}')
print(f'Diesel ₹/ton-km: ₹{diesel_ton_km:,.2f}')
print(f'EV ₹/ton-km:     ₹{ev_ton_km:,.2f}')

## 4. Annual OPEX, TCO, Payback and ROI Proxy

In [ ]:
diesel_annual_opex = diesel_opex_km * annual_km
ev_annual_opex = ev_opex_km * annual_km

diesel_initial = diesel_purchase
ev_initial = ev_purchase + charger_capex_per_ev
incremental_investment = ev_initial - diesel_initial

annual_savings = diesel_annual_opex - ev_annual_opex
payback_years = incremental_investment / annual_savings

diesel_residual = diesel_purchase * residual_pct
ev_residual = ev_purchase * residual_pct

diesel_tco_5y = diesel_initial + diesel_annual_opex * years - diesel_residual
ev_tco_5y = ev_initial + ev_annual_opex * years - ev_residual

roi_proxy = ((annual_savings * years) - incremental_investment) / incremental_investment

summary = pd.DataFrame({
    'Metric': [
        'Annual OPEX / truck', 'Annual OPEX savings / truck',
        'Additional EV investment / truck', 'Simple payback (years)',
        '5-year TCO', '5-year TCO savings', '5-year ROI proxy'
    ],
    'Diesel': [diesel_annual_opex, None, None, None, diesel_tco_5y, None, None],
    'Electric': [ev_annual_opex, annual_savings, incremental_investment,
                payback_years, ev_tco_5y, diesel_tco_5y - ev_tco_5y, roi_proxy]
})
summary

## 5. Fleet Scenarios — 10, 100 and 1,000 Trucks

In [ ]:
fleet_sizes = [10, 100, 1000]

fleet = pd.DataFrame({
    'Fleet size': fleet_sizes,
    'Diesel initial investment': [diesel_initial*n for n in fleet_sizes],
    'EV initial investment': [ev_initial*n for n in fleet_sizes],
    'Diesel annual OPEX': [diesel_annual_opex*n for n in fleet_sizes],
    'EV annual OPEX': [ev_annual_opex*n for n in fleet_sizes],
    'Annual OPEX savings': [annual_savings*n for n in fleet_sizes],
    'Incremental EV investment': [incremental_investment*n for n in fleet_sizes],
})
fleet['Payback years'] = fleet['Incremental EV investment'] / fleet['Annual OPEX savings']
fleet

## 6. Electricity Price Sensitivity

This tests how EV economics change if the electricity price varies.

In [ ]:
prices = [6, 7, 8, 8.27, 9, 10, 11]
sens = []

for p in prices:
    ev_energy_cost = p * ev_energy_km
    ev_cost_km = ev_energy_cost + ev_maintenance + ev_other_opex
    savings = diesel_annual_opex - ev_cost_km * annual_km
    sens.append({
        'Electricity price': p,
        'EV cost/km': ev_cost_km,
        'Annual savings': savings,
        'Payback years': incremental_investment / savings
    })

sens = pd.DataFrame(sens)
sens

## 7. Visualizations

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(['Diesel', 'Electric'], [diesel_opex_km, ev_opex_km])
plt.ylabel('₹ per km')
plt.title('Operating Cost per km')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sens['Electricity price'], sens['Payback years'], marker='o')
plt.xlabel('Electricity price (₹/kWh)')
plt.ylabel('Simple payback (years)')
plt.title('Payback Sensitivity to Electricity Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Interpretation

Use the tables and charts above to answer:

1. Is the EV cheaper on a ₹/km basis under the chosen assumptions?
2. Is the EV cheaper on a ₹/ton-km basis?
3. How many years are required to recover the additional EV investment?
4. How do the economics change at 10, 100 and 1,000 trucks?
5. How sensitive is payback to electricity price?

**Important:** These are scenario results, not a commercial investment recommendation. Replace assumptions with validated OEM quotes, actual route data, charging costs, financing costs, taxes, insurance, driver costs, downtime, battery degradation and applicable regulations for a real-world business case.